# Day 1 — SQL Foundations to Advanced Querying


**Database:** MySQL 8.x. The tables/data are created in a separate setup SQL file; this notebook focuses on syllabus, commands, explanations, practice queries, assignments, and solutions.


## Learning method

1. Run each query manually in MySQL Workbench/CLI.
2. Before executing, predict the result.
3. Change one condition and rerun.
4. Check execution plans where applicable.
5. Complete assignments without looking at the solution first.


## Important execution commands

```sql
SHOW DATABASES;
USE company_sql_practice;
SHOW TABLES;
SELECT DATABASE();
DESCRIBE employees;
```

The setup file contains the five-table schema and large sample dataset.


## Syllabus

- SQL execution order and relational thinking
- SELECT, DISTINCT, aliases, expressions
- WHERE, BETWEEN, IN, LIKE, IS NULL
- CASE expressions
- ORDER BY, LIMIT/OFFSET
- Aggregate functions: COUNT, SUM, AVG, MIN, MAX
- GROUP BY and HAVING
- NULL handling with COALESCE and NULLIF
- Subqueries: scalar, single-row, multi-row, correlated
- CTEs (WITH) and readable query design
- UNION / UNION ALL / INTERSECT / EXCEPT
- EXISTS / NOT EXISTS
- Date and string functions
- Business-style employee/sales questions


## Practice Queries

Each query is intentionally written as a workplace-style example. Read the explanation before changing it.


### Practice 1

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Inspect available objects
SHOW DATABASES;
USE company_sql_practice;
SHOW TABLES;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 2

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Basic filtering
SELECT employee_id, employee_name, department_id, salary
FROM employees
WHERE salary >= 80000
ORDER BY salary DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 3

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- CASE: salary bands
SELECT employee_id, employee_name, salary,
       CASE
         WHEN salary >= 150000 THEN 'Executive'
         WHEN salary >= 100000 THEN 'Senior'
         WHEN salary >= 70000 THEN 'Mid'
         ELSE 'Junior'
       END AS salary_band
FROM employees;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 4

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Aggregation
SELECT department_id,
       COUNT(*) AS employee_count,
       ROUND(AVG(salary), 2) AS avg_salary,
       SUM(salary) AS payroll
FROM employees
GROUP BY department_id
HAVING COUNT(*) >= 2
ORDER BY payroll DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 5

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- CTE
WITH dept_stats AS (
    SELECT department_id,
           COUNT(*) AS employee_count,
           AVG(salary) AS avg_salary
    FROM employees
    GROUP BY department_id
)
SELECT *
FROM dept_stats
WHERE avg_salary > 80000
ORDER BY avg_salary DESC;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 6

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Correlated subquery: employees above their department average
SELECT e.employee_id, e.employee_name, e.department_id, e.salary
FROM employees e
WHERE e.salary > (
    SELECT AVG(e2.salary)
    FROM employees e2
    WHERE e2.department_id = e.department_id
);


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 7

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- EXISTS: employees who have at least one performance review
SELECT e.employee_id, e.employee_name
FROM employees e
WHERE EXISTS (
    SELECT 1
    FROM performance_reviews p
    WHERE p.employee_id = e.employee_id
);


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 8

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- UNION ALL
SELECT employee_id, employee_name, 'Employee' AS source
FROM employees
UNION ALL
SELECT customer_id, customer_name, 'Customer' AS source
FROM customers;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


## Assignments

1. Find the top 10 highest-paid active employees.
2. Find departments with payroll above the company average department payroll.
3. Find employees hired in the last 3 years.
4. Find employees whose salary is above their department average.
5. Find customers who have never placed an order.
6. Find the second-highest distinct salary without using LIMIT.


## Solutions

Try the assignment first. Then compare with the reference pattern below. Adapt names/filters to your dataset.


### Solution 1


In [ ]:
SELECT employee_id, employee_name, salary
FROM employees
WHERE status = 'Active'
ORDER BY salary DESC
LIMIT 10;


### Solution 2


In [ ]:
WITH dept_payroll AS (
  SELECT department_id, SUM(salary) payroll
  FROM employees GROUP BY department_id
)
SELECT * FROM dept_payroll
WHERE payroll > (SELECT AVG(payroll) FROM dept_payroll);


### Solution 3


In [ ]:
SELECT employee_id, employee_name, hire_date
FROM employees
WHERE hire_date >= CURRENT_DATE - INTERVAL 3 YEAR;


### Solution 4


In [ ]:
SELECT e.employee_id, e.employee_name, e.salary
FROM employees e
WHERE e.salary > (
  SELECT AVG(e2.salary) FROM employees e2
  WHERE e2.department_id = e.department_id
);


### Solution 5


In [ ]:
SELECT c.customer_id, c.customer_name
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
WHERE o.order_id IS NULL;


### Solution 6


In [ ]:
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);


## Interview Questions & Answers

**Q: WHERE vs HAVING?**  
A: WHERE filters rows before grouping; HAVING filters grouped results.

**Q: RANK vs DENSE_RANK?**  
A: Both assign equal ranks to ties; RANK leaves gaps after ties, DENSE_RANK does not.

**Q: Why can joins create duplicates?**  
A: A one-to-many relationship legitimately creates multiple result rows per parent. First define the intended row grain, then aggregate/deduplicate as required.

**Q: Cursor or set-based SQL?**  
A: Prefer set-based operations for bulk relational transformations; use cursors when row-by-row procedural behavior is genuinely required.

**Q: What does EXPLAIN provide?**  
A: It exposes the optimizer's planned access path, including table access, candidate indexes, selected index, estimated rows and other plan details.


## 5-Day Deliverable Checklist

- [ ] All practice queries executed manually
- [ ] All assignments solved without copying
- [ ] Solutions compared and rewritten
- [ ] At least 10 query variations created
- [ ] At least 5 window-function problems solved
- [ ] At least 3 stored procedures created
- [ ] At least 1 cursor + handler implemented
- [ ] Transactions tested with COMMIT and ROLLBACK
- [ ] EXPLAIN used before/after an index
- [ ] Final capstone query set completed
